# Phase 2 — Analyse Exploratoire des Données (EDA) : AI4I 2020

Ce notebook explore le jeu de données brut **AI4I 2020 Predictive Maintenance Dataset** (UCI ID `601`), sauvegardé dans `data/raw/ai4i2020.csv`.

### Objectifs de cette exploration
1. Vérifier la qualité des données (dimensions, types, valeurs manquantes, doublons).
2. Analyser la variable cible (`Machine failure`) et les 5 modes de panne (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`).
3. Étudier les distributions des capteurs selon l'état normal ou en panne.
4. Mettre en évidence les interactions physiques utiles pour la future ingénierie de variables (*feature engineering*).
5. Identifier clairement les **limites du dataset** (notamment vis-à-vis du calcul de la durée de vie restante **RUL**).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

# Localise le fichier brut que le notebook soit lancé depuis notebooks/ ou la racine
candidate_paths = [
    Path("../data/raw/ai4i2020.csv"),
    Path("data/raw/ai4i2020.csv"),
]
data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Fichier data/raw/ai4i2020.csv introuvable. Exécutez d'abord src/ml/download_data.py."
    )

df = pd.read_csv(data_path)
print(f"Dataset chargé depuis : {data_path.resolve()}")
print(f"Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes")
df.head()

## 1. Audit de qualité et structure des variables

Vérifions les types de colonnes, l'absence de valeurs manquantes (`NaN`), les doublons et les statistiques descriptives de base.

In [ ]:
print("=== Types des colonnes ===")
print(df.dtypes)

print("\n=== Valeurs manquantes par colonne ===")
print(df.isna().sum())

print(f"\nNombre de lignes dupliquées : {df.duplicated().sum()}")

sensor_cols = [
    "Air temperature",
    "Process temperature",
    "Rotational speed",
    "Torque",
    "Tool wear",
]
df[sensor_cols].describe().round(2)

## 2. Analyse de la variable cible et des 5 modes de panne

- `Machine failure` : cible binaire principale (`0` = normal, `1` = panne).
- `TWF` (*Tool Wear Failure*) : usure excessive de l'outil.
- `HDF` (*Heat Dissipation Failure*) : défaut de dissipation thermique.
- `PWF` (*Power Failure*) : puissance mécanique hors plage tolérée.
- `OSF` (*Overstrain Failure*) : surcharge mécanique (couple × usure).
- `RNF` (*Random Failures*) : défaillance aléatoire.

**Point de vigilance :** vérifions s'il existe un fort **déséquilibre de classes** et si `Machine failure` correspond toujours exactement aux 5 colonnes de modes.

In [ ]:
failure_counts = df["Machine failure"].value_counts().sort_index()
failure_rate = df["Machine failure"].mean() * 100
print(f"Taux global de panne : {failure_rate:.2f}% ({failure_counts[1]} pannes sur {len(df)} observations)")

mode_cols = ["TWF", "HDF", "PWF", "OSF", "RNF"]
mode_sums = df[mode_cols].sum().sort_values(ascending=False)
print("\nNombre d'occurrences par mode de panne :")
print(mode_sums)

no_mode_failures = df[(df["Machine failure"] == 1) & (df[mode_cols].sum(axis=1) == 0)]
rnf_without_failure = df[(df["Machine failure"] == 0) & (df["RNF"] == 1)]
print(f"\nPannes (Machine failure=1) sans aucun mode activé : {len(no_mode_failures)}")
print(f"Lignes avec RNF=1 mais Machine failure=0 : {len(rnf_without_failure)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x="Machine failure", hue="Machine failure", legend=False, ax=axes[0])
axes[0].set_title("Répartition Machine failure (0 = Normal, 1 = Panne)")
axes[0].set_ylabel("Nombre d'observations")

sns.barplot(x=mode_sums.index, y=mode_sums.values, hue=mode_sums.index, legend=False, ax=axes[1])
axes[1].set_title("Occurrences des 5 modes de panne")
axes[1].set_ylabel("Nombre d'occurrences")
axes[1].set_xlabel("Mode de panne")

plt.tight_layout()
plt.show()

## 3. Analyse par type de qualité produit et distribution des capteurs

Comparons le taux de panne selon le type de produit (`L` = Low, `M` = Medium, `H` = High), puis observons comment les valeurs des 5 capteurs varient entre fonctionnement normal et panne.

In [ ]:
type_summary = df.groupby("Type")["Machine failure"].agg(
    observations="count",
    pannes="sum",
    taux_panne=lambda s: round(s.mean() * 100, 2),
)
print("=== Taux de panne par type de qualité ===")
print(type_summary)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for idx, col in enumerate(sensor_cols):
    sns.boxplot(data=df, x="Machine failure", y=col, hue="Machine failure", legend=False, ax=axes[idx])
    axes[idx].set_title(f"{col} selon Machine failure")
    axes[idx].set_xlabel("Machine failure (0 = Normal, 1 = Panne)")

sns.barplot(
    x=type_summary.index,
    y=type_summary["taux_panne"],
    hue=type_summary.index,
    legend=False,
    ax=axes[5],
)
axes[5].set_title("Taux de panne (%) par Type de produit")
axes[5].set_ylabel("Taux de panne (%)")

plt.tight_layout()
plt.show()

## 4. Corrélations et grandeurs physiques combinées

Les pannes industrielles dépendent rarement d'un seul capteur pris isolément ; elles résultent souvent d'interactions physiques :
- **Écart thermique (`Delta_T`) :** `Process temperature - Air temperature` (lié au mode `HDF`).
- **Puissance mécanique (`Power_W`) :** $\text{Couple} \times \text{vitesse angulaire}$, avec $\omega = \text{Rotational speed} \times \frac{2\pi}{60}$ (lié au mode `PWF`).
- **Contrainte d'usure (`Wear_Torque`) :** `Tool wear * Torque` (lié au mode `OSF`).

*(Ces nouvelles variables seront créées proprement lors de la Phase 3 de prétraitement ; ici nous les visualisons uniquement en exploration sans modifier le dataset brut.)*

In [ ]:
numeric_cols = sensor_cols + ["Machine failure"]
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matrice de corrélation (capteurs et panne globale)")
plt.tight_layout()
plt.show()

df_explore = df.copy()
df_explore["Delta_T"] = df_explore["Process temperature"] - df_explore["Air temperature"]
df_explore["Power_W"] = df_explore["Torque"] * (df_explore["Rotational speed"] * 2 * np.pi / 60)
df_explore["Wear_Torque"] = df_explore["Tool wear"] * df_explore["Torque"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.scatterplot(
    data=df_explore, x="Rotational speed", y="Delta_T",
    hue="HDF", alpha=0.6, ax=axes[0]
)
axes[0].set_title("Dissipation thermique (HDF) : Vitesse vs Delta_T")

sns.scatterplot(
    data=df_explore, x="Rotational speed", y="Torque",
    hue="PWF", alpha=0.6, ax=axes[1]
)
axes[1].set_title("Défaut de puissance (PWF) : Vitesse vs Couple")

sns.scatterplot(
    data=df_explore, x="Tool wear", y="Torque",
    hue="OSF", alpha=0.6, ax=axes[2]
)
axes[2].set_title("Surcharge mécanique (OSF) : Usure vs Couple")

plt.tight_layout()
plt.show()

## 5. Synthèse et limites du dataset AI4I 2020

1. **Propreté :** 10 000 lignes complètes, aucune valeur manquante, aucun doublon.
2. **Déséquilibre de classes :** seulement 3,39 % de pannes (`339` sur `10 000`). En Phase 4, l'*accuracy* sera trompeuse ; il faudra privilégier le **Rappel (Recall)**, la **Précision**, le **F1-score** et l'**AUC-PR / ROC-AUC**.
3. **Risque de fuite de données (*data leakage*) :**
   - `UID` et `Product ID` sont des identifiants (à exclure de la modélisation).
   - `TWF`, `HDF`, `PWF`, `OSF`, `RNF` sont des étiquettes de diagnostic (à ne pas mettre en entrée $X$ d'un modèle prédisant `Machine failure`).
4. **Incohérence mineure des labels d'origine :** `9` pannes (`Machine failure=1`) n'ont aucun mode activé, et `18` lignes ont `RNF=1` avec `Machine failure=0`.
5. **Limite majeure concernant le RUL (durée de vie restante) :**
   - AI4I 2020 ne contient **pas de trajectoires temporelles par machine** du début de vie jusqu'à la panne (seulement une séquence synthétique sur un outil partagé).
   - Il est idéal pour la **détection d'anomalies**, la **classification de pannes** et l'**interprétation par agent IA**, mais **ne permet pas d'estimer un RUL réaliste**. Pour cela, il faudra étudier le dataset **NASA C-MAPSS** (Sous-étape 2.2).